## Silver Layer — Transformação

Este notebook é responsável pela camada Silver do pipeline. Ele recebe os dados brutos da camada Bronze e aplica uma série de transformações para deixá-los limpos, padronizados e enriquecidos.

Neste notebook vamos corrigir tipos de dados, remover registros inconsistentes, padronizar as colunas binárias, criar algumas features derivadas úteis para prever churn, normalizar as colunas numéricas e salvar o resultado em Parquet.

In [1]:
""" Imports """
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
""" Constants """
BRONZE_INPUT = Path("../bronze/telco_churn_bronze.parquet")
SILVER_DIR = Path("../silver")
SILVER_OUTPUT = SILVER_DIR / "telco_churn_silver.parquet"

## 1. Carregamento dos Dados Bronze

Os dados são lidos do arquivo Parquet gerado pelo notebook de ingestão (`01_ingestion.ipynb`). Nenhuma transformação é feita aqui — o objetivo é apenas trazer o dado cru para a memória.

In [3]:
if not BRONZE_INPUT.exists():
    raise FileNotFoundError(
        f"Arquivo Bronze não encontrado: {BRONZE_INPUT}\n"
        "Execute o notebook 01_ingestion.ipynb primeiro."
    )

df = pd.read_parquet(BRONZE_INPUT)

print(f"Bronze carregado: {df.shape[0]:,} linhas × {df.shape[1]} colunas")

Bronze carregado: 7,043 linhas × 21 colunas


## 2. Auditoria de Qualidade dos Dados

Antes de transformar qualquer coisa, vamos inspecionar o dado bruto para entender quais problemas existem e justificar cada decisão que vem a seguir.

In [4]:
print("=== Tipos de dados ===")
df.info()

print("\n=== Nulos por coluna ===")
print(df.isnull().sum())

print(f"\n=== Strings vazias em TotalCharges ===")
blank_charges = (df["TotalCharges"] == " ").sum()
print(f"  Linhas com TotalCharges vazias: {blank_charges}")
print(f"  Essas linhas têm tenure=0: {df[df['TotalCharges'] == ' ']['tenure'].unique()}")

=== Tipos de dados ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7

Com a auditoria acima, identificamos três problemas principais. A coluna `TotalCharges` veio como texto em vez de número porque existem 11 linhas com string vazia — são clientes com `tenure = 0` que nunca foram cobrados. Além disso, a coluna `SeniorCitizen` já está em formato numérico (0/1) enquanto todas as outras binárias usam "Yes"/"No", o que é uma inconsistência que vamos resolver.

## 3. Correção de Tipos de Dados

`TotalCharges` deveria ser numérica, mas o pandas a leu como texto por causa das strings vazias. Limpamos os espaços em branco, convertemos as strings vazias para `NaN` e depois fazemos o cast para `float64`.

In [5]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].replace(" ", np.nan), errors="coerce")

print(f"TotalCharges dtype: {df['TotalCharges'].dtype}")
print(f"Nulos em TotalCharges: {df['TotalCharges'].isnull().sum()}")

TotalCharges dtype: float64
Nulos em TotalCharges: 11


## 4. Remoção de Nulos

As 11 linhas com `TotalCharges` nulo são clientes que acabaram de entrar no serviço e nunca foram cobrados. O sinal de churn deles é ambíguo e os dados de cobrança estão ausentes por design, então removê-los é a decisão mais segura.

In [6]:
linhas_antes = len(df)
df = df.dropna(subset=["TotalCharges"])
linhas_depois = len(df)

print(f"Linhas antes: {linhas_antes:,}")
print(f"Linhas removidas: {linhas_antes - linhas_depois}")
print(f"Linhas após drop: {linhas_depois:,}")

Linhas antes: 7,043
Linhas removidas: 11
Linhas após drop: 7,032


## 5. Padronização das Colunas Binárias (Yes/No → 0/1)

`SeniorCitizen` já estava como 0/1, mas `Partner`, `Dependents`, `PhoneService`, `PaperlessBilling`, `Churn` e `gender` usavam strings. Convertemos tudo para inteiro para ter um schema consistente.

Chave: `gender` → Male=1, Female=0. Demais colunas → Yes=1, No=0.

In [7]:
# Colunas com Yes/No
yes_no_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling", "Churn"]
for col in yes_no_cols:
    df[col] = (df[col] == "Yes").astype(int)

# gender: Male=1, Female=0
df["gender"] = (df["gender"] == "Male").astype(int)

print("Colunas binárias padronizadas para 0/1:")
print(df[["gender"] + yes_no_cols].head())

Colunas binárias padronizadas para 0/1:
   gender  Partner  Dependents  PhoneService  PaperlessBilling  Churn
0       0        1           0             0                 1      0
1       1        0           0             1                 0      0
2       1        0           0             1                 1      1
3       1        0           0             0                 0      0
4       0        0           0             1                 1      1


## 6. Colapso das Colunas de Serviços

As colunas de serviços adicionais (`MultipleLines`, `OnlineSecurity`, etc.) têm três valores possíveis: "Yes", "No" e "No phone service" / "No internet service". Esse terceiro valor é na prática equivalente a "No" — se o cliente não tem o serviço base, ele também não tem o adicional. Simplificamos para Yes=1 e qualquer outra coisa=0.

In [8]:
service_cols = [
    "MultipleLines", "OnlineSecurity", "OnlineBackup",
    "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"
]
for col in service_cols:
    df[col] = (df[col] == "Yes").astype(int)

print("Colunas de serviços colapsadas para 0/1:")
print(df[service_cols].head())

Colunas de serviços colapsadas para 0/1:
   MultipleLines  OnlineSecurity  OnlineBackup  DeviceProtection  TechSupport  \
0              0               0             1                 0            0   
1              0               1             0                 1            0   
2              0               1             1                 0            0   
3              0               1             0                 1            1   
4              0               0             0                 0            0   

   StreamingTV  StreamingMovies  
0            0                0  
1            0                0  
2            0                0  
3            0                0  
4            0                0  


## 7. Colunas Derivadas

Criamos três novas features a partir dos dados existentes. Essas colunas não estavam no dataset original — elas são construídas com base no conhecimento do domínio de churn em telecomunicações.

### 7.1 `tenure_group` — Grupo de Ciclo de Vida

A taxa de churn não é linear com o tempo de contrato: clientes novos cancelam muito mais do que clientes antigos. Em vez de tratar `tenure` apenas como número contínuo, agrupamos em quatro estágios do ciclo de vida do cliente (0-12 meses, 13-24, 25-48 e 49-72).

In [9]:
bins = [0, 12, 24, 48, 72]
labels = ["0-12m", "13-24m", "25-48m", "49-72m"]
df["tenure_group"] = pd.cut(
    df["tenure"], bins=bins, labels=labels, right=True, include_lowest=True
).astype(str)

print("Distribuição por grupo de tenure:")
print(df["tenure_group"].value_counts().sort_index())

Distribuição por grupo de tenure:
tenure_group
0-12m     2175
13-24m    1024
25-48m    1594
49-72m    2239
Name: count, dtype: int64


### 7.2 `charges_per_month_ratio` — Gasto Histórico Médio

Calculamos o gasto médio mensal histórico dividindo o total gasto pelo tempo de contrato. Isso funciona como um proxy para detectar mudanças de plano ou descontos ao longo do tempo — se esse valor for bem diferente do `MonthlyCharges` atual, algo mudou na relação do cliente com o serviço.

In [10]:
df["charges_per_month_ratio"] = df["TotalCharges"] / (df["tenure"] + 1)

print("charges_per_month_ratio — estatísticas:")
print(df["charges_per_month_ratio"].describe().round(2))

charges_per_month_ratio — estatísticas:
count    7032.00
mean       59.08
std        30.51
min         9.18
25%        26.23
50%        61.07
75%        84.88
max       118.97
Name: charges_per_month_ratio, dtype: float64


### 7.3 `has_internet_addons` — Possui Serviços Adicionais

Clientes que assinam mais serviços junto com a internet (segurança online, backup, suporte técnico, streaming) tendem a cancelar menos — o bundle cria mais dependência. Essa feature resume em um único bit se o cliente tem ao menos um desses adicionais contratado.

In [11]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
              "TechSupport", "StreamingTV", "StreamingMovies"]
df["has_internet_addons"] = (df[addon_cols].sum(axis=1) > 0).astype(int)

print("Distribuição de has_internet_addons:")
print(df["has_internet_addons"].value_counts())

Distribuição de has_internet_addons:
has_internet_addons
1    4819
0    2213
Name: count, dtype: int64


## 8. Normalização Min-Max das Colunas Contínuas

Escalamos `tenure`, `MonthlyCharges` e `TotalCharges` para o intervalo [0, 1] usando normalização min-max. Escolhemos min-max em vez de z-score porque essas colunas não seguem distribuição normal — `tenure` é quase uniforme e as cobranças são assimétricas. Os valores originais são preservados nas colunas sem sufixo; as versões normalizadas recebem `_norm`.

In [12]:
continuous_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
for col in continuous_cols:
    col_min = df[col].min()
    col_max = df[col].max()
    df[f"{col}_norm"] = (df[col].values - col_min) / (col_max - col_min)

print("Colunas normalizadas (min=0, max=1):")
print(df[[f"{c}_norm" for c in continuous_cols]].describe().round(3))

Colunas normalizadas (min=0, max=1):
       tenure_norm  MonthlyCharges_norm  TotalCharges_norm
count     7032.000             7032.000           7032.000
mean         0.443                0.463              0.261
std          0.346                0.299              0.262
min          0.000                0.000              0.000
25%          0.113                0.173              0.044
50%          0.394                0.518              0.159
75%          0.761                0.713              0.436
max          1.000                1.000              1.000


## 9. Validação do Output

Antes de salvar, checamos que o dataframe não tem nulos e inspecionamos o schema final para confirmar que todas as transformações foram aplicadas corretamente.

In [13]:
assert df.isnull().sum().sum() == 0, "Nulos encontrados no Silver!"

print(f"Shape final: {df.shape[0]:,} linhas × {df.shape[1]} colunas")
print("\nSchema Silver:")
print(df.dtypes)
print("\nAmostra:")
df.head()

Shape final: 7,032 linhas × 27 colunas

Schema Silver:
customerID                  object
gender                       int64
SeniorCitizen                int64
Partner                      int64
Dependents                   int64
tenure                       int64
PhoneService                 int64
MultipleLines                int64
InternetService             object
OnlineSecurity               int64
OnlineBackup                 int64
DeviceProtection             int64
TechSupport                  int64
StreamingTV                  int64
StreamingMovies              int64
Contract                    object
PaperlessBilling             int64
PaymentMethod               object
MonthlyCharges             float64
TotalCharges               float64
Churn                        int64
tenure_group                object
charges_per_month_ratio    float64
has_internet_addons          int64
tenure_norm                float64
MonthlyCharges_norm        float64
TotalCharges_norm          float64


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,charges_per_month_ratio,has_internet_addons,tenure_norm,MonthlyCharges_norm,TotalCharges_norm
0,7590-VHVEG,0,0,1,0,1,0,0,DSL,0,...,Electronic check,29.85,29.85,0,0-12m,14.925000,1,0.000000,0.115423,0.001275
1,5575-GNVDE,1,0,0,0,34,1,0,DSL,1,...,Mailed check,56.95,1889.50,0,25-48m,53.985714,1,0.464789,0.385075,0.215867
2,3668-QPYBK,1,0,0,0,2,1,0,DSL,1,...,Mailed check,53.85,108.15,1,0-12m,36.050000,1,0.014085,0.354229,0.010310
3,7795-CFOCW,1,0,0,0,45,0,0,DSL,1,...,Bank transfer (automatic),42.30,1840.75,0,25-48m,40.016304,1,0.619718,0.239303,0.210241
4,9237-HQITU,0,0,0,0,2,1,0,Fiber optic,0,...,Electronic check,70.70,151.65,1,0-12m,50.550000,0,0.014085,0.521891,0.015330


## 10. Persistência em Parquet

Salvamos o dataframe Silver em formato Parquet usando PyArrow. Preferimos Parquet ao CSV porque ele preserva os tipos de dados que corrigimos ao longo do notebook — se salvássemos em CSV, o pandas leria tudo como texto na próxima vez e perderíamos todo o trabalho de tipagem.

In [14]:
SILVER_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(SILVER_OUTPUT, index=False, engine="pyarrow")

print(f"Silver salvo em: {SILVER_OUTPUT}")
print(f"Tamanho do arquivo: {SILVER_OUTPUT.stat().st_size / 1024:.1f} KB")

Silver salvo em: ../silver/telco_churn_silver.parquet
Tamanho do arquivo: 334.5 KB
